In [ ]:
import os
import re
import torch
import datetime
import librosa
import numpy as np
import soundfile as sf
import wave
from huggingface_hub import snapshot_download, login
from pathlib import Path
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio
import soundfile as sf
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
HF_TOKEN = os.environ.get('HUGGINGFACEHUB_API_TOKEN')
login(token=HF_TOKEN)

In [ ]:
def get_model_path(model_type: str, model_size: str) -> str:
    """Get model path based on type and size."""
    return snapshot_download(f"Qwen/Qwen3-TTS-12Hz-{model_size}-{model_type}")

def write_sound_to_file(
    output_filename: str,
    audio_file: np.array,
    sample_rate: int,
    ) -> None:

    # Write the file
    # subtype='PCM_16' ensures it's a standard WAV file compatible with all players
    sf.write(
      output_filename,
      audio_file,
      sample_rate,
      subtype='PCM_16',
      )

def generate_voice_design(
  text: str,
  language: str,
  voice_description: str,
  ):
    """Generate speech using Voice Design model (1.7B only)."""

    if not text or not text.strip():
        return None, "Error: Text is required."
    if not voice_description or not voice_description.strip():
        return None, "Error: Voice description is required."

    try:
        wavs, sr = voice_design_model.generate_voice_design(
            text=text.strip(),
            language=language,
            instruct=voice_description.strip(),
            non_streaming_mode=True,
            max_new_tokens=2048,
        )
        print("Voice design generation completed successfully!")

        generated_audio_dir = Path("./audio/designed_audio/")
        generated_audio_dir.mkdir(parents=True, exist_ok=True)
        output_filename = generated_audio_dir / "generated_audio.wav"

        # Write the output to file
        write_sound_to_file(
            output_filename = output_filename,
            audio_file = wavs[0],
            sample_rate = sr,
            )

        print(f"Voice design generation written to file {str(output_filename)}")

        #return (sr, wavs[0])
    except Exception as e:
        return None, f"Error: {type(e).__name__}: {e}"

    # Return the file to Weave op
    return {"audio": wave.open(str(output_filename), "rb"), "sample_rate": sr, "audio_array": wavs[0]}


In [ ]:
# Load the Model
try: 
  print(voice_design_model)
except:
  voice_design_model = Qwen3TTSModel.from_pretrained(
      get_model_path("VoiceDesign", "1.7B"),
      device_map=torch.device("cuda") if torch.cuda.is_available() else torch.device("mps"),
      dtype=torch.bfloat16,
      token=HF_TOKEN,
      #attn_implementation="kernels-community/flash-attn3",
  )

In [ ]:
design_text = "You wouldn't suspect little old me, would you?"
design_language = "Auto"
design_instruct = "Use a playful, seductive and mature female voice. The vocal signature should be female but deeper."

designed_voice_output = generate_voice_design(
    text = design_text,
    language=design_language,
    voice_description=design_instruct,
)

In [ ]:
# Generated Audio
display(
    Audio(
        designed_voice_output["audio_array"],
        rate=designed_voice_output["sample_rate"],
        autoplay=False
        )
    )

In [ ]:
# Save generated Audio

output_dir = Path("../data")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "generated_audio.wav"

sf.write(
    output_path,
    designed_voice_output["audio_array"],
    designed_voice_output["sample_rate"],
    subtype='PCM_16',
)

print(f"Saved to {output_path}")